# NewFiguresAug26 — calculations

Heavy computation for the Aug-26 figure set. **Writes** `.npz` files under
`figure_data/aug26_figs/{allosteric,auxetic}/`; the companion notebook
`NewFiguresAug26_plot.ipynb` only reads those files and draws.

Spec: `docs/figureDescription_Aug26.md`. Shared loaders / physics wrappers live in
`aug26_common.py` (imported as `C`).

**Data source** — `data/allosteric_nets_aug` (post-2026-08-20 pipeline) and the `jax`
auxetic tree `data/auxetic_nets_aug/<family>`.

**On faithfulness of the recomputed loss** — the local pull
does *not* include most `timestep_sweep.npz`, so every sweep-derived quantity here is
recomputed from `stiffnesses_traj.npy`. Training ran with **LAMMPS**
(`training_meta.json`). If the `lammps` module is importable in this kernel the
recompute uses it (matches training to ~1e-10); otherwise it falls back to the
JAX-FIRE backend and the *loss-faithfulness* section below reports the discrepancy
so you can decide whether to trust the fallback for a given panel.

Run order: **Setup → Ensemble discovery → Loss-faithfulness → Figure 1 … Figure 5**.
Each figure section is guarded by a `DO_FIGx` flag and skips work whose `.npz`
already exists unless `FORCE_RECOMPUTE`.

In [ ]:
import importlib, sys, time, copy
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path.cwd()))            # so `import aug26_common` works from this dir
import aug26_common as C
importlib.reload(C)

np.set_printoptions(linewidth=140, suppress=False)
print("figure_data ->", C.FIG_DATA_DIR)
print("allosteric tree exists:", C.ALLO_DATA_DIR.is_dir(),
      "| auxetic tree exists:", C.AUX_DATA_DIR.is_dir())

In [ ]:
# ============================ USER KNOBS ============================ #
# Every knob below can be overridden from the environment (AUG26_<NAME>) so the
# notebook can be executed non-interactively for a quick plumbing check, e.g.
#   AUG26_MAX_REAL_PER_TASK_ALLO=1 AUG26_DO_FIG2=0 jupyter nbconvert --execute ...
import os as _os
def _envflag(name, default):
    v = _os.environ.get("AUG26_" + name)
    return default if v is None else v not in ("0", "false", "False", "")
def _envint(name, default):
    v = _os.environ.get("AUG26_" + name)
    return default if v is None else (None if v.lower() == "none" else int(v))

FORCE_RECOMPUTE = _envflag("FORCE_RECOMPUTE", False)

DO_FIG1 = _envflag("DO_FIG1", True)
DO_FIG2 = _envflag("DO_FIG2", True)
DO_FIG3 = _envflag("DO_FIG3", True)
DO_FIG4 = _envflag("DO_FIG4", True)
DO_FIG5 = _envflag("DO_FIG5", True)

DO_ALLOSTERIC = _envflag("DO_ALLOSTERIC", True)
DO_AUXETIC    = _envflag("DO_AUXETIC", True)

# Ensemble size caps (per (geometry, task)).  None = every converged realization.
# Start small — the cost-Hessian / diabatic-tracking loops are the expensive part.
MAX_REAL_PER_TASK_ALLO = _envint("MAX_REAL_PER_TASK_ALLO", 2)
MAX_REAL_PER_TASK_AUX   = _envint("MAX_REAL_PER_TASK_AUX", 1)

# Representative targeted realizations for the single-network panels
# (Fig 1a/c, Fig 2a/b, Fig 3a, Fig 4, Fig 5a-c).
REP_ALLO = ("targeted", 0, 0)          # (geometry_id, task_id, real_id)
REP_ALLO_GEOM_TASKS = [("targeted", t, 0) for t in range(5)]   # Fig 1b: 5 tasks, same geometry
REP_AUX = (1, 0)                        # (task_id, real_id) — task 1 has 2 subtasks
REP_AUX_TASKS = [(t, 0) for t in range(5)]                     # Fig 1d

# Physics backend for the *faithful* recompute.  "auto" -> lammps if importable else jax_fire.
SOLVER_PREF = "auto"

# Cost-Hessian: keep small while validating, raise for the real figures.
C.CFG.k_cost_eigs = _envint("K_COST_EIGS", 6)
C.CFG.n_thresh_steps = _envint("N_THRESH_STEPS", 30)

print("knobs set; CFG =", C.CFG)

## Ensemble discovery

In [ ]:
def resolve_solver(pref, want_lammps_default="lammps"):
    if pref == "auto":
        return "lammps" if C.allo_solver_available("lammps") else "jax_fire"
    return pref

ALLO_SOLVER = resolve_solver(SOLVER_PREF)
print("allosteric recompute solver:", ALLO_SOLVER,
      "" if ALLO_SOLVER == "lammps" else "  (LAMMPS not importable — fallback; see faithfulness check)")

ENS_ALLO = {
    "targeted": C.discover_allosteric("targeted", MAX_REAL_PER_TASK_ALLO),
    "general":  C.discover_allosteric("general",  MAX_REAL_PER_TASK_ALLO),
}
ENS_AUX = C.discover_auxetic(MAX_REAL_PER_TASK_AUX)

for k, v in ENS_ALLO.items():
    print(f"allosteric {k:9s}: {len(v):3d} converged realizations  e.g. {v[:3]}")
print(f"auxetic            : {len(ENS_AUX):3d} converged realizations  e.g. {ENS_AUX[:3]}")
print("representative allo:", REP_ALLO, "| representative aux:", REP_AUX)

# --- Figure 2 c/d (allosteric) and g/h (auxetic) pool -----------------------
# Spec decision (2026-09-01): these four ensemble panels use *every* targeted
# realization that passes the convergence bar (C.MIN_SUCCESS_RATIO = 1e-4) --
# NOT the small MAX_REAL_PER_TASK_* caps above (those still bound the cheap
# ensembles + Fig 3b / Fig 5d) and NOT the `general` geometries. So the pool is
# unambiguously "all targeted tasks, all quality-checked realizations".
FIG2_ALLO_POOL = C.discover_allosteric("targeted", None)          # ~50 realizations, 5 tasks
FIG2_AUX_POOL  = C.discover_auxetic(None)                         # all converged targeted auxetic
print(f"Fig 2 c/d pool  (all converged targeted allosteric): {len(FIG2_ALLO_POOL)}")
print(f"Fig 2 g/h pool  (all converged targeted auxetic)   : {len(FIG2_AUX_POOL)}")


def cached_cost_hessian(ids, k_vec, label, k_eigs):
    """Per-subtask cost Hessian for realization `ids` at stiffness `k_vec`.

    Delegates to `C.allo_cost_hessian_at`, which returns the eigen-pairs
    straight from the realization's `timestep_sweep.npz` when `k_vec` is that
    sweep's before/after checkpoint (per-subtask + combined keys, 2026-08-27+),
    and only otherwise recomputes locally (autodiff FIRE ramp + Lanczos, one
    long `jax.jit` compile per network). Local recomputes are cached to
    `figure_data/aug26_figs/allosteric/_cache/` under `label` (e.g. 'best',
    'init', 'ckpt37'); sweep-served results are cheap enough to re-slice.
    Returns `{"mse1": (evals, evecs), "mse2": (evals, evecs)}` (largest-eig first).
    """
    g, t, r = ids
    key = f"costH_g{g}_t{t}_r{r}_{label}_k{k_eigs}"
    hit = C.cache_get("allo", key)
    if hit is not None:
        return {"mse1": (hit["e1"], hit["v1"]), "mse2": (hit["e2"], hit["v2"])}
    dd = C.load_allosteric(*ids)
    ch = C.allo_cost_hessian_at(dd, k_vec, k_eigs=k_eigs)
    C.cache_put("allo", key, e1=ch["mse1"][0], v1=ch["mse1"][1],
                e2=ch["mse2"][0], v2=ch["mse2"][1])
    return ch


def cached_aux_cost_hessian(task_id, real_id, subtask_idx, k_eigs):
    """`C.aux_cost_hessian_at` for one auxetic subtask (sweep-first when the
    realization's `timestep_sweep.npz` + `stiffness_trajectory.npy` let the
    operating point be matched, else `analysis.cost_utils.compute_cost_hessian`),
    cached to `figure_data/aug26_figs/auxetic/_cache/`. Returns `(evals, evecs)`.
    """
    key = f"costH_aux_t{task_id}_r{real_id}_s{subtask_idx}_k{k_eigs}"
    hit = C.cache_get("aux", key)
    if hit is not None:
        return hit["e"], hit["v"]
    aa = C.load_auxetic(task_id, real_id)
    subs = C.aux_subtasks(aa["task_config"])
    if subtask_idx >= len(subs):
        return None, None
    cs, tp = subs[subtask_idx]
    evals, evecs = C.aux_cost_hessian_at(aa, subtask_idx, cs, tp, k_eigs=k_eigs)
    C.cache_put("aux", key, e=evals, v=evecs)
    return evals, evecs


def _aux_edge_sshift(aa):
    """|s_shift| per edge for an auxetic realization dict (unconstrained inverse)."""
    edges = np.asarray(aa["network"].edges)
    pos = np.asarray(aa["network"].positions)
    rl = np.linalg.norm(pos[edges[:, 1]] - pos[edges[:, 0]], axis=1)
    sus = C.edge_susceptibilities(pos, edges, np.asarray(aa["network"].stiffnesses), rl,
                                  constrained_nodes=None)
    return np.abs(sus["s_shift"])

## Loss-faithfulness check

For a sample of realizations: select the log-spaced-by-loss checkpoints
(`C.select_steps`), recompute the combined loss `0.5*(mse1+mse2)` at the first and
best selected checkpoint with `ALLO_SOLVER`, and compare to the stored value.
Writes `loss_faithfulness_allo.npz` / `_aux.npz`. A large relative error with the
`jax_fire` fallback is expected near convergence (sub-mm differences in output
distance squared blow up in relative terms); with `lammps` it should be ~1e-3 or better.

In [ ]:
def allo_recompute_combined(d, k):
    m0, _ = C.allo_actuation_frames(d["nodes"], d["incidence_matrix"], k, d["task_config"], 0, solver=ALLO_SOLVER)
    m1, _ = C.allo_actuation_frames(d["nodes"], d["incidence_matrix"], k, d["task_config"], 1, solver=ALLO_SOLVER)
    return 0.5 * (m0 + m1)

if DO_ALLOSTERIC:
    rows = []
    sample = ENS_ALLO["targeted"][:6] + ENS_ALLO["general"][:4]
    for ids in sample:
        d = C.load_allosteric(*ids)
        if d is None:
            continue
        ti, lck = C.select_steps(d["mse1"], d["mse2"], d["stiff_traj_steps"])
        for tag, t in (("init", ti[0]), ("best", ti[-1])):
            saved = float(lck[t]) / 2.0        # lck is (mse1+mse2); halve to match the trainer's bar
            recomp = allo_recompute_combined(d, d["stiff_traj"][t])
            rel = abs(recomp - saved) / max(saved, 1e-300)
            rows.append((str(ids), tag, saved, recomp, rel))
            print(f"  {str(ids):20s} {tag}  saved={saved:.4e}  recomp={recomp:.4e}  rel={rel:.3f}")
    rows = np.array(rows, dtype=object)
    C.save_fig_data("allo", "loss_faithfulness_allo",
                    ids=rows[:, 0].astype(str), tag=rows[:, 1].astype(str),
                    saved=rows[:, 2].astype(float), recomp=rows[:, 3].astype(float),
                    rel=rows[:, 4].astype(float), solver=np.array(ALLO_SOLVER))
    rel = rows[:, 4].astype(float)
    tag = rows[:, 1].astype(str)
    print(f"\nmedian relative error: all={np.median(rel):.3f}  "
          f"init-only={np.median(rel[tag=='init']):.3f}  best-only={np.median(rel[tag=='best']):.3f}  "
          f"(solver={ALLO_SOLVER})")
    if ALLO_SOLVER == "lammps":
        assert np.median(rel) < 0.05, "LAMMPS recompute deviates from stored loss — investigate before trusting Fig 2+"
    else:
        print(
            "  >>> WARNING: LAMMPS unavailable — losses were recomputed with JAX-FIRE.\n"
            "      At the *initial* stiffnesses the two solvers agree to <1%, but near\n"
            "      convergence they land at slightly different equilibria and the tiny\n"
            "      absolute output-distance differences blow up in *relative* loss terms\n"
            "      (see the 'best' rows above). Elastic-Hessian eigenvalues and the\n"
            "      susceptibility panels are robust to this; treat absolute cost-Hessian\n"
            "      magnitudes and any 'best-loss' actuated state as approximate. Re-run this\n"
            "      notebook in a kernel with `lammps` importable for a faithful Fig 2/3/4."
        )

In [ ]:
# The auxetic loss-faithfulness recompute JIT-compiles training's JAX Poisson
# solver over the full strain ramp -- a one-off ~20-30 min CPU compile. Set
# AUG26_DO_LOSS_FAITHFULNESS=0 to skip it (the check is independent of every
# figure panel and its .npz, loss_faithfulness_aux.npz, is already on disk).
DO_LOSS_FAITHFULNESS = _envflag("DO_LOSS_FAITHFULNESS", True)

if DO_LOSS_FAITHFULNESS and DO_AUXETIC and ENS_AUX:
    rows = []
    for ids in ENS_AUX[:6]:
        a = C.load_auxetic(*ids)
        if a is None or a["loss_traj"] is None:
            continue
        saved_final = float(a["loss_traj"].min())
        nS = a["task_config"].get("n_strain_steps", 200)
        # faithful recompute: training's own JAX Poisson solver, mean over subtasks
        recs = [C.aux_recompute_loss(a["network"], a["boundary"], cs, tp, nS)
                for cs, tp in C.aux_subtasks(a["task_config"])]
        recomp = float(np.mean(recs))
        rows.append((str(ids), saved_final, recomp, abs(recomp - saved_final) / max(saved_final, 1e-300)))
        print(f"  {str(ids):12s} saved(min loss traj)={saved_final:.4e}  recomp(final K, JAX Poisson)={recomp:.4e}  rel={rows[-1][3]:.3f}")
    if rows:
        rows = np.array(rows, dtype=object)
        C.save_fig_data("aux", "loss_faithfulness_aux",
                        ids=rows[:, 0].astype(str), saved=rows[:, 1].astype(float),
                        recomp=rows[:, 2].astype(float), rel=rows[:, 3].astype(float))
elif DO_AUXETIC:
    print("auxetic loss-faithfulness skipped (AUG26_DO_LOSS_FAITHFULNESS=0)")

## Figure 1 — networks & input/output response

* **1a/1c** one well-trained targeted network, nodes at rest, edges colored by stiffness (magma).
* **1b/1d** input→output response for **5 tasks trained on the same geometry**
  (allosteric: output strain vs input-strain fraction, deep-compression subtask;
   auxetic: Poisson ratio vs applied vertical strain).

Saves: `fig1_allo_network.npz`, `fig1_allo_response.npz`, and the auxetic analogues.

In [ ]:
def need(kind, name):
    return FORCE_RECOMPUTE or not C.fig_data_exists(kind, name)

if DO_FIG1 and DO_ALLOSTERIC:
    # 1a — representative network
    if need("allo", "fig1_allo_network"):
        d = C.load_allosteric(*REP_ALLO)
        k_show = d["best_stiffnesses"] if d["best_stiffnesses"] is not None else d["stiffnesses"]
        C.save_fig_data("allo", "fig1_allo_network",
                        nodes=d["nodes"], edges=d["edges"], stiffnesses=k_show,
                        ids=np.array(REP_ALLO, dtype=object))
        print("fig1_allo_network saved")

    # 1b — 5 tasks, same geometry: deep-compression subtask response.
    # First converged realization of each task in the representative geometry.
    if need("allo", "fig1_allo_response"):
        rep_geom = REP_ALLO[0]
        by_task = {}
        for (g, t, r) in ENS_ALLO["targeted"] if rep_geom == "targeted" else ENS_ALLO["general"]:
            if g == rep_geom:
                by_task.setdefault(t, (g, t, r))
        tasks_to_show = [by_task[t] for t in sorted(by_task)][:5]
        curves = []
        for ids in tasks_to_show:
            d = C.load_allosteric(*ids)
            if d is None:
                print("  missing", ids); continue
            k = d["best_stiffnesses"] if d["best_stiffnesses"] is not None else d["stiffnesses"]
            _, frames = C.allo_actuation_frames(d["nodes"], d["incidence_matrix"], k,
                                                d["task_config"], 0, solver=ALLO_SOLVER)
            d23_0 = np.linalg.norm(d["nodes"][3] - d["nodes"][2])
            out_strain = np.array([np.linalg.norm(f[3] - f[2]) / d23_0 - 1.0 for f in frames])
            in_frac = np.linspace(1.0 / len(frames), 1.0, len(frames))
            curves.append(dict(ids=ids, in_frac=in_frac, out_strain=out_strain,
                               target=d["task_config"]["strain_output"],
                               strain_input=d["task_config"]["strain_input"]))
        C.save_fig_data("allo", "fig1_allo_response",
                        curves=np.array(curves, dtype=object))
        print(f"fig1_allo_response saved ({len(curves)} tasks)")

In [ ]:
if DO_FIG1 and DO_AUXETIC:
    if need("aux", "fig1_aux_network"):
        a = C.load_auxetic(*REP_AUX)
        C.save_fig_data("aux", "fig1_aux_network",
                        nodes=np.asarray(a["network"].positions), edges=np.asarray(a["network"].edges),
                        stiffnesses=np.asarray(a["network"].stiffnesses), ids=np.array(REP_AUX, dtype=object))
        print("fig1_aux_network saved")

    if need("aux", "fig1_aux_response"):
        curves = []
        for ids in REP_AUX_TASKS:
            a = C.load_auxetic(*ids)
            if a is None:
                print("  missing", ids); continue
            cs_deep, tp_deep = C.aux_subtasks(a["task_config"])[0]
            frames = C.aux_compression_frames(a["network"], a["boundary"], cs_deep,
                                              n_steps=a["task_config"].get("n_strain_steps", 100))
            eps_yy, nu = C.poisson_along_traj(frames, a["boundary"])
            curves.append(dict(ids=ids, eps_yy=eps_yy, nu=nu, target=tp_deep, compression=cs_deep))
        C.save_fig_data("aux", "fig1_aux_response", curves=np.array(curves, dtype=object))
        print(f"fig1_aux_response saved ({len(curves)} tasks)")

## Figure 2 — Hessian structure vs training / actuation

* **2a** eigenvalues of the **elastic Hessian of the network at rest** (undeformed
  node positions, no actuation) as a function of `L / L_init`, at every selected
  log-spaced-by-loss checkpoint, representative task. One faint line per eigen-branch.
  *(Replaces the old per-subtask cost-Hessian panel — cheap, so every `select_steps`
  checkpoint is kept; both the constrained (nodes 0-3 pinned) and unconstrained
  spectra are stored, the plot shows the constrained one.)*
* **2b** **constrained elastic Hessian** eigenvalues vs the **actual imposed strain**,
  along **one** actuation trajectory — the **deepest subtask only** (its ramp already
  spans the shallower subtask's imposed-strain range), representative task. Two curve
  sets: `before` training and `best`-loss step. `*_strain` is the imposed input-strain
  magnitude at each stored frame (strain fraction × that subtask's target strain);
  `strain_shallow` marks where the shallow subtask's ramp ends (plot draws a dotted line).
* **2c** distributions of the **ratio** `λ0(after) / λ0(before)` of the lowest
  non-trivial elastic-Hessian eigenvalue, at 3 "compressions" (no actuation /
  subtask-1 end / subtask-2 end), across **all converged targeted realizations**
  (`FIG2_ALLO_POOL`). 'before' = first selected checkpoint, 'after' = best-loss K.
  The no-compression ratio is stored for constrained + unconstrained (plot toggle);
  each subtask's imposed strain is stored so the plot labels by strain, not number.
* **2d** **avoided-crossing trajectory positions** from diabatic tracking
  (`C.detect_avoided_crossings_diabatic`): the normalised position along the
  actuation trajectory `(t + 0.5)/(T - 1)` of every detected event, pooled over both
  subtasks and **all converged targeted realizations**, separately for the before- and
  best-loss stiffnesses. The plot is a histogram over trajectory position (not a count
  per realization). Mirrors the ensemble event-strain histogram in
  `NewFiguresJune_allosteric.ipynb`.

Panels 2c / 2d are the full targeted pool only — one file each,
`fig2c_targeted.npz` / `fig2d_targeted.npz` (no more `_general` variants).

In [ ]:
if DO_FIG2 and DO_ALLOSTERIC:
    d = C.load_allosteric(*REP_ALLO)
    ti, lck = C.select_steps(d["mse1"], d["mse2"], d["stiff_traj_steps"])
    L_init = float(lck[ti[0]])

    # ---- 2a : elastic Hessian of the network AT REST vs training loss ----
    # (was: per-subtask cost-Hessian spectrum). At every selected log-by-loss
    # checkpoint we take that checkpoint's stiffnesses and diagonalise the
    # elastic Hessian at the *undeformed* (rest) node positions -- no actuation,
    # no autodiff -- so keeping every `select_steps` checkpoint is cheap. Both
    # the constrained (nodes 0-3 pinned, as in 2b/2c) and the unconstrained
    # spectra are stored; the plot shows the constrained one.
    if need("allo", "fig2a"):
        rel_L, eig_con, eig_unc = [], [], []
        for j, t in enumerate(ti):
            k_t = d["stiff_traj"][t]
            vc = C.allo_elastic_spectrum(d["nodes"], d["edges"], k_t, d["eq_lengths"],
                                         constrained_nodes=C.ALLO_CONSTRAINED_NODES)[0]
            vu = C.allo_elastic_spectrum(d["nodes"], d["edges"], k_t, d["eq_lengths"],
                                         constrained_nodes=None)[0]
            rel_L.append(float(lck[t]) / L_init)
            eig_con.append(vc)
            eig_unc.append(vu)
        C.save_fig_data("allo", "fig2a",
                        rel_L=np.array(rel_L),
                        eig_constrained=np.array(eig_con),
                        eig_unconstrained=np.array(eig_unc),
                        ids=np.array(REP_ALLO, dtype=object))
        print(f"fig2a saved  ({len(rel_L)} checkpoints, rest-config elastic spectrum "
              f"{np.array(eig_con).shape})")

    # ---- 2b : constrained elastic Hessian vs IMPOSED STRAIN, before vs best-loss ----
    # ONE actuation trajectory only -- the deepest subtask, whose ramp already
    # spans the shallower subtask's imposed-strain range. x-axis = actual imposed
    # input-strain magnitude (strain fraction * that subtask's target strain);
    # `strain_shallow` marks where the shallow subtask's ramp ends. Schema:
    #   before, best              (nH, M) constrained elastic spectra
    #   before_strain, best_strain (nH,)  imposed |strain_in| at each stored frame
    #   strain_shallow            scalar  shallow subtask's target |strain_in|
    if need("allo", "fig2b"):
        s_in0 = abs(float(d["task_config"]["strain_input"]))
        s_in1 = abs(float(d["task_config"]["strain_input2"]))
        si_deep = 0 if s_in0 >= s_in1 else 1
        s_deep, s_shallow = (s_in0, s_in1) if si_deep == 0 else (s_in1, s_in0)
        nH = C.CFG.n_hessian_traj_steps
        out = {"strain_shallow": np.array(s_shallow)}
        for tag, t in (("before", ti[0]), ("best", ti[-1])):
            k_t = d["stiff_traj"][t]
            _, frames = C.allo_actuation_frames(d["nodes"], d["incidence_matrix"], k_t,
                                                d["task_config"], si_deep, solver=ALLO_SOLVER)
            idx = np.round(np.linspace(0, len(frames) - 1, nH)).astype(int)
            specs = [C.allo_elastic_spectrum(frames[i], d["edges"], k_t, d["eq_lengths"])[0]
                     for i in idx]
            out[tag] = np.stack(specs)                          # (nH, M)
            out[tag + "_strain"] = (idx + 1) / len(frames) * s_deep
        C.save_fig_data("allo", "fig2b", ids=np.array(REP_ALLO, dtype=object), **out)
        print(f"fig2b saved  (deepest subtask {si_deep + 1}, imposed |strain_in| 0..{s_deep:g})")

In [ ]:
def allo_lam0_ratio_at_compressions(ids):
    """Ratio of the lowest non-trivial elastic-Hessian eigenvalue AFTER training
    to the same quantity BEFORE training, at three "compressions":
    (0) no actuation / rest config, (1) subtask-1 actuated end state,
    (2) subtask-2 actuated end state.

    'before' stiffnesses = first selected checkpoint (initial);
    'after'  stiffnesses = best-loss (fallback last checkpoint).
    The rest-config ratio is returned for both the constrained (nodes 0-3
    pinned) and the unconstrained Hessian; the actuated-state ratios use the
    constrained Hessian. Also returns the imposed input strain of each subtask
    for plot labelling.
    """
    d = C.load_allosteric(*ids)
    if d is None:
        return None
    ti, lck = C.select_steps(d["mse1"], d["mse2"], d["stiff_traj_steps"])
    k_before = d["stiff_traj"][ti[0]]
    k_after = (d["best_stiffnesses"] if d["best_stiffnesses"] is not None
               else d["stiff_traj"][ti[-1]])

    def _ratio(fn):
        b = fn(k_before)
        a = fn(k_after)
        return a / b if (np.isfinite(b) and b > 0) else np.nan

    out = {}
    for tag, cn, con in (("constrained", C.ALLO_CONSTRAINED_NODES, True),
                         ("unconstrained", None, False)):
        out[f"nocomp_{tag}"] = _ratio(lambda k, cn=cn, con=con: C.lowest_nontrivial(
            C.allo_elastic_spectrum(d["nodes"], d["edges"], k, d["eq_lengths"],
                                    constrained_nodes=cn)[0], constrained=con))
    for subtask in (0, 1):
        def _end_lam0(k, subtask=subtask):
            _, frames = C.allo_actuation_frames(d["nodes"], d["incidence_matrix"], k,
                                                d["task_config"], subtask, solver=ALLO_SOLVER)
            return C.lowest_nontrivial(
                C.allo_elastic_spectrum(frames[-1], d["edges"], k, d["eq_lengths"])[0],
                constrained=True)
        out[f"sub{subtask + 1}"] = _ratio(_end_lam0)
    out["strain_sub1"] = float(d["task_config"]["strain_input"])
    out["strain_sub2"] = float(d["task_config"]["strain_input2"])
    return out


if DO_FIG2 and DO_ALLOSTERIC:
    # Pool = ALL converged targeted realizations (see FIG2_ALLO_POOL in the
    # ensemble-discovery cell). No `general` variant, no per-task cap. Panel c
    # shows the distribution of lambda0(after)/lambda0(before) at each compression.
    name = "fig2c_targeted"
    if need("allo", name):
        acc = {k: [] for k in ("nocomp_constrained", "nocomp_unconstrained", "sub1", "sub2")}
        strn1, strn2 = [], []
        for j, ids in enumerate(FIG2_ALLO_POOL):
            r = allo_lam0_ratio_at_compressions(ids)
            if r is None:
                continue
            for k in acc:
                acc[k].append(r[k])
            strn1.append(r["strain_sub1"]); strn2.append(r["strain_sub2"])
            if (j + 1) % 10 == 0:
                print(f"  2c targeted: {j+1}/{len(FIG2_ALLO_POOL)}", flush=True)
        C.save_fig_data("allo", name,
                        ratio_nocomp_constrained=np.array(acc["nocomp_constrained"]),
                        ratio_nocomp_unconstrained=np.array(acc["nocomp_unconstrained"]),
                        ratio_sub1=np.array(acc["sub1"]), ratio_sub2=np.array(acc["sub2"]),
                        strain_sub1=np.array(strn1), strain_sub2=np.array(strn2))
        print(name, "saved", len(acc["sub1"]), "realizations")

In [ ]:
FIG2D_NSTEPS = _envint("FIG2D_NSTEPS", 100)   # actuation resolution for allosteric avoided-crossing tracking


def allo_avoided_crossing_positions(ids, n_steps=FIG2D_NSTEPS):
    """Normalised actuation-trajectory position `(t + 0.5)/(T - 1)` of every
    diabatic avoided crossing, pooled over both subtasks, for the before- and
    best-loss stiffnesses. Returns `(pos_before, pos_after)` float arrays (one
    entry per detected event), or `None` if the realization is missing.

    The actuation ramp is re-run at `n_steps` resolution (not the training
    `nsteps` 40/20) so the diabatic tracker has enough steps to resolve
    branch-character swaps -- matches `N_STEPS=100` in
    `NewFiguresJune_allosteric.ipynb`.
    """
    d = C.load_allosteric(*ids)
    if d is None:
        return None
    ti, lck = C.select_steps(d["mse1"], d["mse2"], d["stiff_traj_steps"])
    k_before = d["stiff_traj"][ti[0]]
    k_after = (d["best_stiffnesses"] if d["best_stiffnesses"] is not None
               else d["stiff_traj"][ti[-1]])
    tc = dict(d["task_config"]); tc["nsteps"] = n_steps; tc["nsteps2"] = n_steps
    out = {"before": [], "after": []}
    for tag, k in (("before", k_before), ("after", k_after)):
        for subtask in (0, 1):
            _, frames = C.allo_actuation_frames(d["nodes"], d["incidence_matrix"], k,
                                                tc, subtask, solver=ALLO_SOLVER)
            _, vl = C.spectrum_and_lowmodes(frames, k, d["eq_lengths"], d["edges"])
            denom = max(len(frames) - 1, 1)
            out[tag].extend((t + 0.5) / denom
                            for (t, _m) in C.detect_avoided_crossings_diabatic(vl))
    return np.array(out["before"], float), np.array(out["after"], float)


if DO_FIG2 and DO_ALLOSTERIC:
    # Pool + accumulation: every event's trajectory position over ALL converged
    # targeted realizations (FIG2_ALLO_POOL), before vs after training. The plot
    # renders these as a histogram over trajectory position (cf. the ensemble
    # event-strain histogram in NewFiguresJune_allosteric.ipynb), not a per-
    # realization count.
    name = "fig2d_targeted"
    if need("allo", name):
        pos_b, pos_a = [], []
        for j, ids in enumerate(FIG2_ALLO_POOL):
            r = allo_avoided_crossing_positions(ids)
            if r is None:
                continue
            pos_b.extend(r[0].tolist()); pos_a.extend(r[1].tolist())
            if (j + 1) % 10 == 0:
                print(f"  2d targeted: {j+1}/{len(FIG2_ALLO_POOL)}  "
                      f"(events so far: {len(pos_b)} before / {len(pos_a)} after)", flush=True)
        C.save_fig_data("allo", name,
                        pos_before=np.array(pos_b, float),
                        pos_after=np.array(pos_a, float),
                        n_real=np.array(len(FIG2_ALLO_POOL)))
        print(f"{name} saved  ({len(pos_b)} before / {len(pos_a)} after events "
              f"over {len(FIG2_ALLO_POOL)} realizations)")

In [ ]:
# ---- Figure 2 e-h : auxetic mirror ----
FIG2H_NSTEPS = _envint("FIG2H_NSTEPS", 100)   # compression steps for the avoided-crossing trajectories
FIG2G_NSTEPS = _envint("FIG2G_NSTEPS", 100)   # compression steps for the panel-g lambda0-ratio end states


def _aux_checkpoint_indices_losses(a):
    """(t_indices, stored_loss) for the representative auxetic realization:
    the sweep's own log-by-loss selection when a timestep_sweep is present,
    else a log-spaced fallback on the raw loss trajectory."""
    sw = a.get("sweep")
    if sw is not None and "t_indices" in sw and "stored_loss" in sw:
        return np.asarray(sw["t_indices"], int), np.asarray(sw["stored_loss"], float)
    loss = np.asarray(a["loss_traj"], float)
    s = loss.mean(axis=1) if loss.ndim > 1 else loss
    n = min(C.CFG.n_thresh_steps, len(s))
    idx = np.unique(np.round(np.linspace(0, len(s) - 1, n)).astype(int))
    return idx, s[idx]


def _aux_net_with_stiff(aa, k):
    net = copy.deepcopy(aa["network"])
    net.stiffnesses = np.asarray(k, float)
    return net


def aux_avoided_crossing_positions(ids, n_steps=FIG2H_NSTEPS):
    """Normalised compression-trajectory position `(t + 0.5)/(T - 1)` of every
    diabatic avoided crossing, pooled over all subtasks, for the initial and
    trained stiffnesses. Returns `(pos_before, pos_after)` or `None`."""
    aa = C.load_auxetic(*ids)
    if aa is None or aa.get("stiff_traj") is None:
        return None
    st = np.asarray(aa["stiff_traj"])
    k_before = st[0]                                   # untrained initial stiffnesses
    k_after = np.asarray(aa["network"].stiffnesses, float)
    subs = C.aux_subtasks(aa["task_config"])
    out = {"before": [], "after": []}
    for tag, k in (("before", k_before), ("after", k_after)):
        net_k = _aux_net_with_stiff(aa, k)
        for (cs, tp) in subs:
            frames = C.aux_compression_frames(net_k, aa["boundary"], cs, n_steps=n_steps)
            _, vl = C.aux_spectrum_and_lowmodes(frames, net_k, aa["boundary"])
            denom = max(len(frames) - 1, 1)
            out[tag].extend((t + 0.5) / denom
                            for (t, _m) in C.detect_avoided_crossings_diabatic(vl))
    return np.array(out["before"], float), np.array(out["after"], float)


def aux_lam0_ratio_at_compressions(ids, n_steps=FIG2G_NSTEPS):
    """Auxetic analogue of `allo_lam0_ratio_at_compressions`: lambda0(after)/
    lambda0(before) at (0) rest, (1) subtask-1 compression end, (2) subtask-2
    compression end. 'before' = stiffness_trajectory[0], 'after' = trained
    network. Also returns each subtask's imposed compression strain.
    """
    aa = C.load_auxetic(*ids)
    if aa is None or aa.get("stiff_traj") is None:
        return None
    net_b = _aux_net_with_stiff(aa, np.asarray(aa["stiff_traj"])[0])
    net_a = aa["network"]
    pos0 = np.asarray(net_a.positions)

    def _ratio(fb, fa):
        return fa / fb if (np.isfinite(fb) and fb > 0) else np.nan

    out = {}
    for tag, con in (("constrained", True), ("unconstrained", False)):
        fb = C.lowest_nontrivial(C.aux_elastic_spectrum(pos0, net_b, aa["boundary"], constrained=con)[0],
                                 constrained=con)
        fa = C.lowest_nontrivial(C.aux_elastic_spectrum(pos0, net_a, aa["boundary"], constrained=con)[0],
                                 constrained=con)
        out[f"nocomp_{tag}"] = _ratio(fb, fa)
    subs = C.aux_subtasks(aa["task_config"])[:2]
    out["strain_sub1"] = out["strain_sub2"] = np.nan
    for si, (cs, tp) in enumerate(subs):
        frb = C.aux_compression_frames(net_b, aa["boundary"], cs, n_steps=n_steps)
        fra = C.aux_compression_frames(net_a, aa["boundary"], cs, n_steps=n_steps)
        fb = C.lowest_nontrivial(C.aux_elastic_spectrum(frb[-1], net_b, aa["boundary"])[0], constrained=True)
        fa = C.lowest_nontrivial(C.aux_elastic_spectrum(fra[-1], net_a, aa["boundary"])[0], constrained=True)
        out[f"sub{si + 1}"] = _ratio(fb, fa)
        out[f"strain_sub{si + 1}"] = float(cs)
    if "sub1" not in out:
        out["sub1"] = np.nan
    if "sub2" not in out:
        out["sub2"] = np.nan
    return out


if DO_FIG2 and DO_AUXETIC:
    a = C.load_auxetic(*REP_AUX)
    subs = C.aux_subtasks(a["task_config"])
    nS = a["task_config"].get("n_strain_steps", 100)

    # ---- 2f : constrained elastic Hessian vs IMPOSED |compression|, before vs best-loss ----
    # Mirror of allosteric 2b: ONE compression trajectory only -- the deepest
    # subtask (aux_subtasks is sorted by descending |compression|, so subs[0]).
    # x-axis = actual imposed |eps_yy| (strain fraction * deepest |compression|);
    # `strain_shallow` marks the shallow subtask's imposed |compression|. Schema
    # matches allo fig2b: before / best / before_strain / best_strain / strain_shallow.
    # 'before' = initial stiffnesses (stiffness_trajectory[0]); when there is no
    # stiffness trajectory on disk only the 'best' (trained) curve is stored.
    if need("aux", "fig2b"):
        st0 = a.get("stiff_traj")
        net_b = _aux_net_with_stiff(a, np.asarray(st0)[0]) if st0 is not None else None
        cs_deep = abs(float(subs[0][0]))
        cs_shallow = abs(float(subs[1][0])) if len(subs) > 1 else np.nan
        nH = C.CFG.n_hessian_traj_steps
        out = {"strain_shallow": np.array(cs_shallow)}
        pairs = [("best", a["network"])] if net_b is None else [("before", net_b), ("best", a["network"])]
        for tag, net in pairs:
            frames = C.aux_compression_frames(net, a["boundary"], subs[0][0], n_steps=nS)
            idx = np.round(np.linspace(0, len(frames) - 1, nH)).astype(int)
            specs = np.stack([C.aux_elastic_spectrum(frames[i], net, a["boundary"])[0] for i in idx])
            out[tag] = specs                                    # (nH, M)
            out[tag + "_strain"] = (idx + 1) / len(frames) * cs_deep
        C.save_fig_data("aux", "fig2b", ids=np.array(REP_AUX, dtype=object), **out)
        print("aux fig2b saved  "
              + (f"(before + best-loss, deepest subtask, |eps_yy| 0..{cs_deep:g})"
                 if net_b is not None
                 else f"(best-loss only -- no stiffness trajectory on disk, |eps_yy| 0..{cs_deep:g})"))

    # ---- 2e : elastic Hessian of the network AT REST vs training loss ----
    if need("aux", "fig2a"):
        if a.get("stiff_traj") is not None:
            ti_a, L_a = _aux_checkpoint_indices_losses(a)
            st = np.asarray(a["stiff_traj"])
            pos0 = np.asarray(a["network"].positions)
            rel_L, eig_con, eig_unc = [], [], []
            for t, Lt in zip(ti_a, L_a):
                if not (0 <= int(t) < len(st)):
                    continue
                net_k = _aux_net_with_stiff(a, st[int(t)])
                eig_con.append(C.aux_elastic_spectrum(pos0, net_k, a["boundary"], constrained=True)[0])
                eig_unc.append(C.aux_elastic_spectrum(pos0, net_k, a["boundary"], constrained=False)[0])
                rel_L.append(float(Lt) / float(L_a[0]))
            C.save_fig_data("aux", "fig2a",
                            rel_L=np.array(rel_L),
                            eig_constrained=np.array(eig_con),
                            eig_unconstrained=np.array(eig_unc),
                            ids=np.array(REP_AUX, dtype=object))
            print(f"aux fig2a saved  ({len(rel_L)} checkpoints, rest-config elastic spectrum)")
        else:
            print("aux fig2a skipped: no stiffness trajectory on disk")

    # ---- 2g : lambda0(after)/lambda0(before) ratio at 3 compressions, ALL targeted auxetic ----
    if need("aux", "fig2c_targeted"):
        acc = {k: [] for k in ("nocomp_constrained", "nocomp_unconstrained", "sub1", "sub2")}
        strn1, strn2 = [], []
        for j, ids in enumerate(FIG2_AUX_POOL):
            r = aux_lam0_ratio_at_compressions(ids)
            if r is None:
                continue
            for k in acc:
                acc[k].append(r[k])
            strn1.append(r.get("strain_sub1", np.nan)); strn2.append(r.get("strain_sub2", np.nan))
            if (j + 1) % 20 == 0:
                print(f"  2g targeted (aux): {j+1}/{len(FIG2_AUX_POOL)}", flush=True)
        C.save_fig_data("aux", "fig2c_targeted",
                        ratio_nocomp_constrained=np.array(acc["nocomp_constrained"]),
                        ratio_nocomp_unconstrained=np.array(acc["nocomp_unconstrained"]),
                        ratio_sub1=np.array(acc["sub1"]), ratio_sub2=np.array(acc["sub2"]),
                        strain_sub1=np.array(strn1), strain_sub2=np.array(strn2))
        print("aux fig2c_targeted saved", len(acc["sub1"]), "realizations")

    # ---- 2h : avoided-crossing trajectory positions, ALL targeted auxetic ----
    if need("aux", "fig2d_targeted"):
        pos_b, pos_a = [], []
        for j, ids in enumerate(FIG2_AUX_POOL):
            try:
                r = aux_avoided_crossing_positions(ids, n_steps=FIG2H_NSTEPS)
            except Exception as e:
                print(f"  skip {ids}: {e}", flush=True)
                continue
            if r is None:
                continue
            pos_b.extend(r[0].tolist()); pos_a.extend(r[1].tolist())
            if (j + 1) % 10 == 0:
                print(f"  2h targeted (aux): {j+1}/{len(FIG2_AUX_POOL)}  "
                      f"(events so far: {len(pos_b)} before / {len(pos_a)} after)", flush=True)
        C.save_fig_data("aux", "fig2d_targeted",
                        pos_before=np.array(pos_b, float),
                        pos_after=np.array(pos_a, float),
                        n_real=np.array(len(FIG2_AUX_POOL)))
        print(f"aux fig2d_targeted saved  ({len(pos_b)} before / {len(pos_a)} after events "
              f"over {len(FIG2_AUX_POOL)} realizations)")

## Figure 3 — cost-Hessian top modes vs shift susceptibility

* **3a** `|top cost-Hessian eigenvector|` per edge, **for each of the two subtasks** (MSE1 / MSE2),
  plotted against `|s_shift|` per edge. Representative task. The per-subtask top eigenvector
  is read from `timestep_sweep.npz` (`cost_hessian_after_eigvecs`, ascending order -> top =
  last column).
* **3b / 3d** distribution of the Spearman rank correlation between `|v_top|` and `|s_shift|`,
  one value per **(realization x subtask)**, over the **full converged pool** (no cap now
  that the cost Hessian is sweep-served). Allosteric writes both `fig3b_targeted` (50) and
  `fig3b_general` (~200+); the plot's panel **b** shows the *general* pool. Auxetic `3d`
  uses the full `AUX_FAMILY` (default `targeted`) auxetic tree.

Saves `fig3a_allo.npz`, `fig3b_{targeted,general}.npz`, `fig3a_aux.npz`, `fig3b_targeted.npz` (aux).


In [ ]:
# fig3b: distribution of Spearman rho(|v_top|, |s_shift|) over the ensemble.
# The per-subtask cost Hessian is served straight from each realization's
# timestep_sweep.npz (via C.allo_operating_point -> C.allo_cost_hessian_at) as
# long as k_eigs <= the sweep's stored count (4 for allosteric) -- so with
# FIG3B_K_EIGS = 4 this is pure I/O and needs no per-realization cap. Both pools
# are written; the plot picks which one panel 3b shows.
FIG3B_MAX_REAL = _envint("FIG3B_MAX_REAL", None)     # None = every converged realization
FIG3B_K_EIGS = _envint("FIG3B_K_EIGS", 4)            # <= sweep's stored k (allo=4) -> sweep-served

def allo_top_costvec_vs_sshift(ids, at="best", k_eigs=FIG3B_K_EIGS):
    """Per-edge |top cost-Hessian eigenvector| (per subtask) + |s_shift|, cached
    per realization to figure_data/.../allo/_cache/ so a killed 3b run resumes
    for free (each realization is one Box-cold multi-file load + a susceptibility
    solve)."""
    g, t, r = ids
    ckey = f"fig3b_g{g}_t{t}_r{r}_{at}_k{k_eigs}"
    hit = C.cache_get("allo", ckey)
    if hit is not None:
        return {k: hit[k] for k in ("s_shift", "v_mse1", "v_mse2")}
    d = C.load_allosteric(*ids)
    if d is None:
        return None
    k, _tag = C.allo_operating_point(d, at)
    ch = cached_cost_hessian(ids, k, at, k_eigs)
    v1 = np.abs(ch["mse1"][1][:, 0]); v2 = np.abs(ch["mse2"][1][:, 0])   # top eigenvector per subtask
    sus = C.edge_susceptibilities(d["nodes"], d["edges"], k, d["eq_lengths"])
    out = dict(s_shift=np.abs(sus["s_shift"]), v_mse1=v1, v_mse2=v2)
    C.cache_put("allo", ckey, **out)
    return out


if DO_FIG3 and DO_ALLOSTERIC:
    if need("allo", "fig3a_allo"):
        r = allo_top_costvec_vs_sshift(REP_ALLO)
        C.save_fig_data("allo", "fig3a_allo", ids=np.array(REP_ALLO, dtype=object), **r)
        print("fig3a_allo saved")

    # Full converged pools (not the small MAX_REAL_PER_TASK_ALLO caps): 3b should
    # be an actual distribution over subtasks x tasks x realizations.
    FIG3B_POOLS = {
        "targeted": C.discover_allosteric("targeted", None),
        "general":  C.discover_allosteric("general", None),
    }
    for pool, all_ids in FIG3B_POOLS.items():
        name = f"fig3b_{pool}"
        if not need("allo", name):
            continue
        pool_ids = all_ids if FIG3B_MAX_REAL is None else all_ids[:FIG3B_MAX_REAL]
        rhos = []
        for j, ids in enumerate(pool_ids):
            r = allo_top_costvec_vs_sshift(ids)
            if r is None:
                continue
            rhos.append(C.spearman(r["v_mse1"], r["s_shift"]))
            rhos.append(C.spearman(r["v_mse2"], r["s_shift"]))
            if (j + 1) % 25 == 0:
                print(f"  3b {pool}: {j+1}/{len(pool_ids)}", flush=True)
        C.save_fig_data("allo", name, rho=np.array(rhos, dtype=float),
                        n_real=np.array(len(pool_ids)))
        print(name, "saved", np.array(rhos).shape, "over", len(pool_ids), "realizations")


In [ ]:
# ---- 3c : auxetic |top cost-Hessian eigenvector| per subtask vs |s_shift| (representative) ----
def aux_top_costvecs_from_sweep(aa):
    """[|v_top| per edge] for each subtask, read straight from the realization's
    timestep_sweep.npz (`cost_hessian_after_eigvecs`, ascending eigen-order so the
    top mode is the LAST column). Returns None when the sweep is absent."""
    sw = aa.get("sweep")
    if sw is None or "cost_hessian_after_eigvecs" not in sw:
        return None
    V = np.asarray(sw["cost_hessian_after_eigvecs"])        # (n_sub, n_edges, k)
    return [np.abs(V[s, :, -1]) for s in range(V.shape[0])]


if DO_FIG3 and DO_AUXETIC and need("aux", "fig3a_aux"):
    a = C.load_auxetic(*REP_AUX)
    n_sub = len(C.aux_subtasks(a["task_config"]))
    payload = dict(ids=np.array(REP_AUX, dtype=object), s_shift=_aux_edge_sshift(a))
    vs = aux_top_costvecs_from_sweep(a)
    for si in range(min(2, n_sub)):
        if vs is not None and si < len(vs):
            payload[f"v_sub{si + 1}"] = vs[si]
        else:
            _, evecs = cached_aux_cost_hessian(REP_AUX[0], REP_AUX[1], si, C.CFG.k_cost_eigs)
            payload[f"v_sub{si + 1}"] = np.abs(evecs[:, 0])
    C.save_fig_data("aux", "fig3a_aux", **payload)
    print(f"fig3a_aux saved ({min(2, n_sub)} subtask(s))")


In [ ]:
# ---- 3d : auxetic Spearman(|v_top|, |s_shift|) distribution over the ensemble ----
# The per-subtask top cost-Hessian eigenvector is read straight from each
# realization's timestep_sweep.npz (`cost_hessian_after_eigvecs`) -> pure I/O.
# NOTE: only ~22 of the targeted auxetic realizations currently have a sweep
# on disk (mostly task 0); realizations WITHOUT a sweep are skipped rather than
# triggering the multi-minute JAX cost-Hessian retrace. Regenerate the auxetic
# sweeps (analysis.timestep_sweep) for a full-ensemble 3d. Set AUG26_FIG3B_AUX_RECOMPUTE=1
# to fall back to the JAX retrace for the missing ones (slow).
FIG3B_AUX_MAX_REAL = _envint("FIG3B_AUX_MAX_REAL", None)
FIG3B_AUX_RECOMPUTE = _envflag("FIG3B_AUX_RECOMPUTE", False)


def aux_rho_vtop_sshift(tid, rid):
    """[rho(|v_top|, |s_shift|) per subtask] for one auxetic realization, cached
    per realization. Returns [] (skip) when the realization has no timestep_sweep
    and FIG3B_AUX_RECOMPUTE is off."""
    ckey = f"fig3d_t{tid}_r{rid}"
    hit = C.cache_get("aux", ckey)
    if hit is not None:
        return list(np.asarray(hit["rho"], float))
    aa = C.load_auxetic(tid, rid)
    if aa is None:
        return None
    vs = aux_top_costvecs_from_sweep(aa)
    if vs is None:
        if not FIG3B_AUX_RECOMPUTE:
            return []                       # no sweep -> skip (don't JAX-retrace)
        vs = []
        for si in range(min(2, len(C.aux_subtasks(aa["task_config"])))):
            _, evecs = cached_aux_cost_hessian(tid, rid, si, C.CFG.k_cost_eigs)
            vs.append(np.abs(evecs[:, 0]))
    s_shift = _aux_edge_sshift(aa)
    rr = [C.spearman(v, s_shift) for v in vs[:2] if np.asarray(v).shape == s_shift.shape]
    C.cache_put("aux", ckey, rho=np.array(rr, dtype=float))
    return rr


if DO_FIG3 and DO_AUXETIC and need("aux", "fig3b_targeted"):
    all_ids = C.discover_auxetic(None)
    pool_ids = all_ids if FIG3B_AUX_MAX_REAL is None else all_ids[:FIG3B_AUX_MAX_REAL]
    rhos, n_used, n_skipped = [], 0, 0
    for j, (tid, rid) in enumerate(pool_ids):
        rr = aux_rho_vtop_sshift(tid, rid)
        if rr is None:
            continue
        if not rr:                              # no sweep -> skipped
            n_skipped += 1
            continue
        rhos.extend(rr); n_used += 1
        if (j + 1) % 25 == 0:
            print(f"  3d (aux): {j + 1}/{len(pool_ids)}  (used {n_used}, skipped {n_skipped})", flush=True)
    C.save_fig_data("aux", "fig3b_targeted", rho=np.array(rhos, dtype=float),
                    n_real=np.array(n_used), n_skipped=np.array(n_skipped))
    print(f"fig3b_targeted (aux) saved {np.array(rhos).shape} over {n_used} sweep-served "
          f"realizations ({n_skipped} skipped for lack of a sweep)")


## Figure 4 — Fig3Maker panels, one allosteric + one auxetic targeted task

Port of the panels in `Fig3Maker050526.ipynb` for a single representative task, assembled
into one matplotlib figure on the plot side. That legacy notebook computes, at fine
resolution along the actuation ramp:

1. elastic-Hessian **spectrum vs input strain** (`FreeSpectrum050526.pdf`);
2. **mode overlaps** `|<du | v_k>|` between the incremental displacement and each Hessian
   eigenvector, near a chosen operating strain.

This section reproduces (1) and (2) with the repo's current physics wrappers.
Saves `fig4_allo.npz` / `fig4_aux.npz`.

In [ ]:
FIG4_N_STRAIN = 120     # fine ramp resolution for the spectrum-vs-strain panel

if DO_FIG4 and DO_ALLOSTERIC:
    if need("allo", "fig4_allo"):
        d = C.load_allosteric(*REP_ALLO)
        ti, _ = C.select_steps(d["mse1"], d["mse2"], d["stiff_traj_steps"])
        k = d["best_stiffnesses"] if d["best_stiffnesses"] is not None else d["stiff_traj"][ti[-1]]
        tc = dict(d["task_config"]); tc["nsteps"] = FIG4_N_STRAIN
        _, frames = C.allo_actuation_frames(d["nodes"], d["incidence_matrix"], k, tc, 0, solver=ALLO_SOLVER)
        d01 = np.linalg.norm(d["nodes"][0] - d["nodes"][1])
        eps_in = np.array([np.linalg.norm(f[1] - f[0]) / d01 - 1.0 for f in frames])
        specs, overlaps = [], []
        prev = None
        for i, f in enumerate(frames):
            vals, vecs = C.allo_elastic_spectrum(f, d["edges"], k, d["eq_lengths"])
            specs.append(vals)
            if prev is not None:
                du = (f - prev).ravel()
                # project onto free-DOF eigenvectors (vecs already free-DOF space, zero-padded)
                du_free = du[np.setdiff1d(np.arange(2 * len(f)), np.r_[C.ALLO_CONSTRAINED_NODES * 2,
                                                                       C.ALLO_CONSTRAINED_NODES * 2 + 1])]
                n = np.linalg.norm(du_free)
                overlaps.append(np.abs(vecs.T @ (du_free / n)) if n > 1e-14 else np.zeros(vecs.shape[1]))
            prev = f
        C.save_fig_data("allo", "fig4_allo", ids=np.array(REP_ALLO, dtype=object),
                        eps_in=eps_in, spectrum=np.stack(specs),
                        overlaps=np.stack(overlaps), nodes=d["nodes"], edges=d["edges"], stiffnesses=k)
        print("fig4_allo saved  spectrum", np.stack(specs).shape)

if DO_FIG4 and DO_AUXETIC:
    if need("aux", "fig4_aux"):
        a = C.load_auxetic(*REP_AUX)
        cs_deep, tp_deep = C.aux_subtasks(a["task_config"])[0]
        frames = C.aux_compression_frames(a["network"], a["boundary"], cs_deep, n_steps=FIG4_N_STRAIN)
        eps_yy, nu = C.poisson_along_traj(frames, a["boundary"])
        specs = [C.aux_elastic_spectrum(f, a["network"], a["boundary"])[0] for f in frames]
        C.save_fig_data("aux", "fig4_aux", ids=np.array(REP_AUX, dtype=object),
                        eps_yy=eps_yy, nu=nu, spectrum=np.stack(specs),
                        nodes=np.asarray(a["network"].positions), edges=np.asarray(a["network"].edges),
                        stiffnesses=np.asarray(a["network"].stiffnesses))
        print("fig4_aux saved")

## Figure 5 — susceptibility components vs mechanical quantities

Representative targeted task. Per-edge quantities computed **at each subtask** (two point sets):

* **5a** `s_parallel` vs `|bond strain|`
* **5b** `s_perp` vs `bond stress (sigma)`
* **5c** `s_eq` vs `bond stress (sigma)`
* **5d** heatmap of Spearman correlations across subtasks × tasks × realizations of the
  **general** ensemble: rows `|s_par|, |s_perp|, |s_eq|, |s_shift|` × cols `stiffness, stress, strain`.

Saves `fig5abc_allo.npz`, `fig5d_{pool}.npz` (+ auxetic analogues).

In [ ]:
def allo_edge_table(ids):
    """Per-edge susceptibilities + bond quantities at each subtask's actuated end state."""
    d = C.load_allosteric(*ids)
    if d is None:
        return None
    ti, _ = C.select_steps(d["mse1"], d["mse2"], d["stiff_traj_steps"])
    k = d["best_stiffnesses"] if d["best_stiffnesses"] is not None else d["stiff_traj"][ti[-1]]
    rows = {}
    for subtask in (0, 1):
        _, frames = C.allo_actuation_frames(d["nodes"], d["incidence_matrix"], k,
                                            d["task_config"], subtask, solver=ALLO_SOLVER)
        pos = frames[-1]
        sus = C.edge_susceptibilities(pos, d["edges"], k, d["eq_lengths"])
        bq = C.bond_quantities(pos, d["edges"], k, d["eq_lengths"])
        rows[subtask] = {**sus, **bq}
    return rows


if DO_FIG5 and DO_ALLOSTERIC:
    if need("allo", "fig5abc_allo"):
        rows = allo_edge_table(REP_ALLO)
        payload = {"ids": np.array(REP_ALLO, dtype=object)}
        for subtask in (0, 1):
            for key, val in rows[subtask].items():
                payload[f"sub{subtask}_{key}"] = val
        C.save_fig_data("allo", "fig5abc_allo", **payload)
        print("fig5abc_allo saved")

    for pool in ("targeted", "general"):
        name = f"fig5d_{pool}"
        if not need("allo", name):
            continue
        comps = ["s_par", "s_perp", "s_eq", "s_shift"]
        mechs = ["stiffness", "stress", "strain"]
        acc = {(c, m): [] for c in comps for m in mechs}
        for j, ids in enumerate(ENS_ALLO[pool]):
            rows = allo_edge_table(ids)
            if rows is None:
                continue
            for subtask in (0, 1):
                r = rows[subtask]
                for c in comps:
                    for m in mechs:
                        acc[(c, m)].append(C.spearman(np.abs(r[c]), np.abs(r[m])))
            if (j + 1) % 10 == 0:
                print(f"  5d {pool}: {j+1}/{len(ENS_ALLO[pool])}", flush=True)
        mat = np.array([[np.nanmean(acc[(c, m)]) for m in mechs] for c in comps])
        mat_all = np.array([[np.array(acc[(c, m)], float) for m in mechs] for c in comps], dtype=object)
        C.save_fig_data("allo", name, matrix=mat, rows=np.array(comps), cols=np.array(mechs),
                        per_pair=mat_all)
        print(name, "saved\n", np.round(mat, 3))

In [ ]:
if DO_FIG5 and DO_AUXETIC:
    if need("aux", "fig5abc_aux"):
        a = C.load_auxetic(*REP_AUX)
        payload = {"ids": np.array(REP_AUX, dtype=object)}
        for si, (cs, tp) in enumerate(C.aux_subtasks(a["task_config"])[:2]):
            frames = C.aux_compression_frames(a["network"], a["boundary"], cs,
                                              n_steps=a["task_config"].get("n_strain_steps", 100))
            pos = frames[-1]
            edges = np.asarray(a["network"].edges)
            rl = np.linalg.norm(np.asarray(a["network"].positions)[edges[:, 1]]
                                - np.asarray(a["network"].positions)[edges[:, 0]], axis=1)
            sus = C.edge_susceptibilities(pos, edges, np.asarray(a["network"].stiffnesses), rl,
                                          constrained_nodes=None)
            bq = C.bond_quantities(pos, edges, np.asarray(a["network"].stiffnesses), rl)
            for key, val in {**sus, **bq}.items():
                payload[f"sub{si}_{key}"] = val
        C.save_fig_data("aux", "fig5abc_aux", **payload)
        print("fig5abc_aux saved")

---
### Done

All requested `.npz` files are under `figure_data/aug26_figs/`. Open
`NewFiguresAug26_plot.ipynb` to render.

**Known gaps / TODO** (search the code above for `TODO` / `NOTE`):
* Fig 2b / 2f now store a **single deepest-subtask trajectory** vs the actual
  imposed strain (`before` / `best` / `*_strain` / `strain_shallow`). The plot
  still accepts the old per-subtask schema (`sub0_*`) as a fallback, so stale
  `fig2b.npz` files render without a re-run — delete them (or `FORCE_RECOMPUTE`)
  to pick up the new x-axis.
* Fig 2 c/d (allosteric) and g/h (auxetic) use the **full converged targeted
  pool** (`FIG2_ALLO_POOL` / `FIG2_AUX_POOL`), not the `MAX_REAL_PER_TASK_*`
  caps. The auxetic 2h loop is the slow one — `FIG2H_NSTEPS` (env
  `AUG26_FIG2H_NSTEPS`) sets the compression resolution used for diabatic
  tracking.
* if `lammps` is not importable here, Fig 2+ used the JAX-FIRE fallback — check
  `loss_faithfulness_allo.npz` before trusting absolute eigenvalue magnitudes.